In [2]:
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO

import os
import cv2
import torch
import torchvision
import torchvision.transforms as T
import numpy as np
from tqdm import tqdm

# Define transformations (optional)
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch.nn as nn
import torch.nn.functional as F

In [3]:
# Path to the annotation file
annotation_file = "/kaggle/input/segmentation-dataset/result.json"
image_folder = "/kaggle/input/segmentation-dataset/"

# Load COCO annotations
coco = COCO(annotation_file)

# Get all image IDs
image_ids = coco.getImgIds()

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


In [4]:
class COCOSegmentationDataset(Dataset):
    def __init__(self, coco, image_folder, transforms=None):
        self.coco = coco
        self.image_folder = image_folder
        self.image_ids = coco.getImgIds()
        self.transforms = transforms

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id, iscrowd=False)
        annotations = self.coco.loadAnns(ann_ids)
        
        # Load image
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.image_folder, img_info["file_name"])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Create segmentation mask
        mask = np.zeros((img_info["height"], img_info["width"]), dtype=np.uint8)
        boxes = []
        labels = []
        
        for ann in annotations:
            segmentation = ann["segmentation"]
            for seg in segmentation:
                pts = np.array(seg, dtype=np.int32).reshape(-1, 2)
                cv2.fillPoly(mask, [pts], color=1)
            
            # Get bounding box in [xmin, ymin, xmax, ymax] format
            x, y, w, h = ann["bbox"]
            boxes.append([x, y, x + w, y + h])
            labels.append(ann["category_id"])
        
        # Convert boxes and labels to tensors
        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        mask = torch.tensor(mask, dtype=torch.uint8)  # This is a PyTorch tensor
        
        # Handle empty annotations
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            mask = torch.zeros((img_info["height"], img_info["width"]), dtype=torch.uint8)
        
        # Correct box shape handling
        boxes = boxes.view(-1, 4)  # Ensure shape is [N, 4]
        
        # Create target dictionary
        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": mask.unsqueeze(0),  # Add batch dimension
        }

        # Apply transformations (only image and mask at first)
        if self.transforms:
            # Convert mask to numpy array before passing to albumentations
            mask_np = mask.numpy()  # Convert to numpy array

            # Convert bounding boxes and labels to lists before passing
            bboxes_list = boxes.tolist()  # Convert tensor to list of lists
            labels_list = labels.tolist()  # Convert tensor to list

            # Normalize bounding box coordinates to [0.0, 1.0] range
            height, width = image.shape[:2]
            bboxes_normalized = [
                [xmin / width, ymin / height, xmax / width, ymax / height]
                for xmin, ymin, xmax, ymax in bboxes_list
            ]

            # Apply transformations for image and mask only (no bboxes or labels yet)
            transformed = self.transforms(image=image, mask=mask_np, bboxes=bboxes_normalized, labels=labels_list)
            
            # Update image and mask after transformation
            image = transformed['image']
            mask = transformed['mask']
            
            # Clip bounding box coordinates to [0.0, 1.0] range
            bboxes_transformed = transformed['bboxes']
            bboxes_clipped = [
                [max(0.0, min(1.0, xmin)), max(0.0, min(1.0, ymin)), max(0.0, min(1.0, xmax)), max(0.0, min(1.0, ymax))]
                for xmin, ymin, xmax, ymax in bboxes_transformed
            ]

            # Update target with transformed bboxes and labels
            target['masks'] = torch.tensor(transformed['mask'], dtype=torch.uint8).unsqueeze(0)
            target['boxes'] = torch.tensor(bboxes_clipped, dtype=torch.float32)
            target['labels'] = torch.tensor(transformed['labels'], dtype=torch.int64)
        
        # Normalize the image to [0, 1] range
        image = image.float() / 255.0  # Convert to float and normalize
        
        # Ensure target['boxes'] is always [N, 4], even if N=0
        if target['boxes'].numel() == 0:
            target['boxes'] = torch.zeros((0, 4), dtype=torch.float32)
            target['labels'] = torch.zeros((0,), dtype=torch.int64)
        
        return image, target

In [5]:
# Define transformations with augmentation
transform = A.Compose([
    A.RandomResizedCrop(height=256, width=256, scale=(0.8, 1.0), p=1),  # Random crop
    A.HorizontalFlip(p=0.5),  # Horizontal flip
    A.RandomBrightnessContrast(p=0.2),  # Random brightness/contrast
    A.HueSaturationValue(p=0.3),  # Random color jitter
    A.RandomRotate90(p=0.5),  # Random rotation (90 degrees)
    A.RandomSizedBBoxSafeCrop(width=256, height=256, p=1),  # Random safe crop
    ToTensorV2(),  # Convert image and mask to tensor
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

# Create dataset instance with augmentation
dataset = COCOSegmentationDataset(coco, image_folder, transforms=transform)

# DataLoader
train_loader = DataLoader(
    dataset, 
    batch_size=1, 
    shuffle=True, 
    num_workers=2,  # Use >0 for parallel data loading
    pin_memory=True  # Optimizes memory transfer for GPU training
)

In [6]:
model_path = '/kaggle/input/ecg-object-detection-model/pytorch/default/1/trained_ecg_object_detection_model.pth'
num_classes = 2 # 1 class (ECG) + background 

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')

# modify weights to pre-trained ones
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
model.load_state_dict(torch.load(model_path))

# Convert from objection detection to segmentation

# Load a Mask R-CNN model with the same backbone
mask_model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')

# Copy the trained Faster R-CNN backbone
mask_model.backbone = model.backbone

# Copy the trained box predictor (classification head)
mask_model.roi_heads.box_predictor = model.roi_heads.box_predictor

# Modify the mask head for segmentation (depends on dataset)
in_features_mask = mask_model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256  # Default value in Mask R-CNN
mask_model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
    in_features_mask, hidden_layer, num_classes
)

<ipython-input-6-916dff75f08e>:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


In [7]:
# Move to GPU if available
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
mask_model.to(device)
print(f"Device: {device}")

Device: cuda


In [8]:
def train_model(model, dataloader, optimizer, num_epochs=10, device="cuda"):
    model.to(device)  # Move model to the correct device
    model.train()  # Set model to training mode
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)
        
        for images, targets in progress_bar:
            # Move images to the correct device
            images = [img.to(device) for img in images]

            # Squeeze unnecessary dimensions for boxes, labels, and masks
            if len(targets["boxes"]) > 0:
                targets["boxes"] = targets["boxes"].squeeze(0)  # Remove batch dimension
                if targets["boxes"].dim() == 1:  # If we end up with a 1D tensor, reshape to [1, 4]
                    targets["boxes"] = targets["boxes"].unsqueeze(0)  # Ensure shape is [1, 4]

                targets["labels"] = targets["labels"].squeeze(0)
                if targets["labels"].dim() == 0:  # If it's a scalar, reshape to [1]
                    targets["labels"] = targets["labels"].unsqueeze(0)

                targets["masks"] = targets["masks"].squeeze(0)
                if targets["masks"].dim() == 2:  # If the mask is 2D, ensure it has the proper batch dimension
                    targets["masks"] = targets["masks"].unsqueeze(0)  # Ensure shape is [1, H, W]
            else:
                print("No bounding boxes in this target!")

            # Move targets to the correct device
            targets = {k: v.to(device) for k, v in targets.items()}

            # Forward pass: Get the loss
            loss_dict = model(images, [targets])  # Wrap targets into a list for a single batch
            loss = sum(loss for loss in loss_dict.values())

            # Backward pass: Compute gradients
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Track the loss
            epoch_loss += loss.item()
            progress_bar.set_postfix(loss=epoch_loss / len(dataloader))
        
        # Print epoch loss
        print(f"Epoch {epoch+1} finished with loss: {epoch_loss / len(dataloader):.4f}")
    
    print("Training complete!")


In [9]:
 # Define optimizer
optimizer = torch.optim.Adam(mask_model.parameters(), lr=0.0001)

# Train model
train_model(mask_model, train_loader, optimizer, num_epochs=1, device=device)

Epoch 1/10:   0%|          | 0/90 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in divide
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in greater_equal
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
<ipython-input-4-1fcf512277e0>:90: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target['masks'] = torch.tensor(transformed['mask'], dtype=torch.uint8).unsqueeze(0)
/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in divide
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
/u

Epoch 1 finished with loss: 0.0164


Epoch 2/10:   0%|          | 0/90 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in divide
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in divide
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in greater_equal
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in greater_equal
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
<ipython-input-4-1fcf512277e0>:90: UserWarning: To copy construct from a tensor, it is recommended to us

Epoch 2 finished with loss: 0.0000


Epoch 3/10:   0%|          | 0/90 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in divide
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in greater_equal
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
<ipython-input-4-1fcf512277e0>:90: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target['masks'] = torch.tensor(transformed['mask'], dtype=torch.uint8).unsqueeze(0)
/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in divide
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
/u

...

Epoch 8 finished with loss: 0.0000


Epoch 9/10:   0%|          | 0/90 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in divide
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in greater_equal
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
<ipython-input-4-1fcf512277e0>:90: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target['masks'] = torch.tensor(transformed['mask'], dtype=torch.uint8).unsqueeze(0)
/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in divide
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
/u

Epoch 9 finished with loss: 0.0001


Epoch 10/10:   0%|          | 0/90 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in divide
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in greater_equal
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
<ipython-input-4-1fcf512277e0>:90: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target['masks'] = torch.tensor(transformed['mask'], dtype=torch.uint8).unsqueeze(0)
/usr/local/lib/python3.10/dist-packages/albumentations/core/bbox_utils.py:478: RuntimeWarning: invalid value encountered in divide
  & (clipped_box_areas / denormalized_box_areas >= min_visibility - epsilon)
/

Epoch 10 finished with loss: 0.0000
Training complete!


In [30]:
torch.save(mask_model.state_dict(), "ecg_segmentation_model.pth")


Test model on new image

In [43]:
import matplotlib.pyplot as plt

mask_model.eval()

# Load and preprocess the image
# image_path = "/kaggle/input/practice-images/train_000152.png"
image_path = "/kaggle/input/practice-images/train_000152.png"
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
resized_image = cv2.resize(image, (256, 256))

transform = transforms.Compose([
    transforms.ToTensor(),
])
input_tensor = transform(resized_image).unsqueeze(0)

# Run inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_tensor = input_tensor.to(device)

In [44]:
with torch.no_grad():
    output = mask_model(input_tensor)

print(output)

[{'boxes': tensor([], device='cuda:0', size=(0, 4)), 'labels': tensor([], device='cuda:0', dtype=torch.int64), 'scores': tensor([], device='cuda:0'), 'masks': tensor([], device='cuda:0', size=(0, 1, 256, 256))}]


In [45]:
predicted_mask = output[0]["masks"].squeeze(0).squeeze(0)
predicted_mask = (predicted_mask > 0.5).float()
predicted_mask_np = predicted_mask.cpu().numpy()

# Resize the mask to the original image size
mask_resized = cv2.resize(predicted_mask_np, (image.shape[1], image.shape[0]))

error: OpenCV(4.10.0) /io/opencv/modules/imgproc/src/resize.cpp:4152: error: (-215:Assertion failed) !ssize.empty() in function 'resize'


In [38]:

# Create an overlay
overlay = image.copy()
overlay[mask_resized == 1] = [255, 0, 0]  # Highlight the mask in red
alpha = 0.5
result = cv2.addWeighted(image, 1 - alpha, overlay, alpha, 0)

# Display the result
plt.imshow(result)
plt.axis("off")
plt.show()

# Save the results
cv2.imwrite("predicted_mask.png", predicted_mask_np * 255)
cv2.imwrite("overlay_result.png", cv2.cvtColor(result, cv2.COLOR_RGB2BGR))

NameError: name 'mask_resized' is not defined